# Proyecto final de Text Mining
## Preparación de la narración

El objetivo es preparar la narración del partido para que pueda ser analizada posteriormente. Se trabajará únicamente con `narracion_original.txt`; el enlace del video se conservará como referencia de procedencia, pero no se descargará ni se utilizará como entrada.

Se conservará el texto original, se normalizarán sus tiempos, se producirá una versión limpia y se documentarán las decisiones tomadas. La ficha corresponde a la transcripción del video: [https://www.youtube.com/watch?v=VMfNgKSGMk4](https://www.youtube.com/watch?v=VMfNgKSGMk4). El partido registrado es Real Madrid 0–4 FC Barcelona, disputado el 26 de octubre de 2024 en el Santiago Bernabéu.


In [ ]:
import json
import os
import re
import unicodedata
from pathlib import Path

import pandas as pd
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_openai import ChatOpenAI

BASE_DIR = Path.cwd()
if not (BASE_DIR / 'narracion_original.txt').exists():
    BASE_DIR = BASE_DIR / 'proyecto_final_text_mining'
ARCHIVO_ORIGINAL = BASE_DIR / 'narracion_original.txt'
MODO_LIMPIEZA = 'demo'

PARTIDO = {
    'equipos': 'Real Madrid vs. FC Barcelona',
    'fecha': '2024-10-26',
    'estadio': 'Santiago Bernabeu',
    'resultado': 'Real Madrid 0-4 FC Barcelona',
    'fuente_efectiva': ARCHIVO_ORIGINAL.name,
    'enlace_referencia': 'https://www.youtube.com/watch?v=VMfNgKSGMk4',
    'alcance': 'Transcripcion local proporcionada en el proyecto; no se descarga audio ni video.'
}

valor_api = os.environ["OPENAI_API_KEY"]
PARTIDO


{'equipos': 'Real Madrid vs. FC Barcelona',
 'fecha': '2024-10-26',
 'estadio': 'Santiago Bernabeu',
 'resultado': 'Real Madrid 0-4 FC Barcelona',
 'fuente_efectiva': 'narracion_original.txt',
 'enlace_referencia': 'https://www.youtube.com/watch?v=VMfNgKSGMk4',
 'alcance': 'Transcripcion local proporcionada en el proyecto; no se descarga audio ni video.'}

### Configuración y alcance

El archivo `.env` contiene únicamente `OPENAI_API_KEY`. La clave no se imprime ni se guarda en el notebook.


In [10]:
def leer_transcripcion(ruta):
    return Path(ruta).read_text(encoding='utf-8').splitlines()

contenido_original = ARCHIVO_ORIGINAL.read_text(encoding='utf-8')
lineas_originales = contenido_original.splitlines()

print(f'Archivo: {ARCHIVO_ORIGINAL.name}')
print(f'Lineas leidas: {len(lineas_originales):,}')
print(f'Palabras aproximadas: {len(contenido_original.split()):,}')

Archivo: narracion_original.txt
Lineas leidas: 819
Palabras aproximadas: 21,269


### Revisión inicial

La transcripción contiene 819 intervenciones. Se observa que la marca temporal y su explicación en lenguaje natural fueron concatenadas, por ejemplo `1:051 minuto y 5 segundos`. Esto no altera la información, pero sí dificulta ordenar o agrupar la narración; por eso se conservarán ambos valores en columnas separadas.


In [11]:
TIME_RE = re.compile(r'^(?P<tc>\d{1,2}:\d{2}(?::\d{2})?)')
UNITS_RE = re.compile(
    r'^\s*(?:\d+\s+hora(?:s)?(?:,\s*)?)?'
    r'(?:\d+\s+minuto(?:s)?(?:\s+y\s*)?)?'
    r'(?:\d+\s+segundo(?:s)?)\s*', re.IGNORECASE
)
MINUTES_ONLY_RE = re.compile(r'^\s*(?:\d+\s+hora(?:s)?(?:,\s*)?)?\d+\s+minuto(?:s)?\s*', re.IGNORECASE)

def timestamp_a_segundos(timestamp):
    partes = [int(p) for p in timestamp.split(':')]
    if len(partes) == 2:
        return partes[0] * 60 + partes[1]
    return partes[0] * 3600 + partes[1] * 60 + partes[2]

def segundos_a_hms(total):
    horas, resto = divmod(int(total), 3600)
    minutos, segundos = divmod(resto, 60)
    return f'{horas:02d}:{minutos:02d}:{segundos:02d}'

def separar_marca_tiempo(linea, numero):
    coincidencia = TIME_RE.match(linea)
    if not coincidencia:
        raise ValueError(f'No se encontro timestamp en la linea {numero}')
    timestamp = coincidencia.group('tc')
    restante = linea[coincidencia.end():]
    texto = UNITS_RE.sub('', restante, count=1)
    if texto == restante:
        texto = MINUTES_ONLY_RE.sub('', restante, count=1)
    segundos = timestamp_a_segundos(timestamp)
    return {
        'linea': numero,
        'timestamp_original': timestamp,
        'tiempo_hms': segundos_a_hms(segundos),
        'segundos': segundos,
        'texto_original': texto.strip()
    }

registros = [separar_marca_tiempo(linea, i) for i, linea in enumerate(lineas_originales, start=1)]
df = pd.DataFrame(registros)
print(df[['linea', 'timestamp_original', 'tiempo_hms', 'segundos']].head(5).to_string(index=False))
print(f'Ultimo tiempo: {df.iloc[-1].tiempo_hms}')


 linea timestamp_original tiempo_hms  segundos
     1               0:00   00:00:00         0
     2               0:08   00:00:08         8
     3               0:10   00:00:10        10
     4               0:19   00:00:19        19
     5               0:25   00:00:25        25
Ultimo tiempo: 01:40:25


### Normalización temporal

La función anterior interpreta el primer código temporal de cada línea y elimina únicamente la descripción redundante de horas, minutos y segundos. El resultado conserva el tiempo original, un formato `HH:MM:SS` y el valor numérico en segundos, que será útil para seleccionar los primeros diez minutos o agrupar eventos.


In [12]:
ALIAS_FUTBOL = {
    'offside': 'fuera de juego'
}

def limpiar_texto_demo(texto):
    texto = unicodedata.normalize('NFC', texto)
    for incorrecto, correcto in ALIAS_FUTBOL.items():
        texto = re.sub(rf'\b{re.escape(incorrecto)}\b', correcto, texto, flags=re.IGNORECASE)
    texto = re.sub(r'\s+', ' ', texto).strip()
    texto = re.sub(r'\s+([,.;:!?])', r'\1', texto)
    texto = re.sub(r'([,.;:!?])(?=\w)', r'\1 ', texto)
    texto = re.sub(r'\b(eh)(?:\s+\1)+\b', '', texto, flags=re.IGNORECASE)
    texto = re.sub(r'\b(de|que|y)(?:\s+\1)+\b', r'\1', texto, flags=re.IGNORECASE)
    return re.sub(r'\s{2,}', ' ', texto).strip()

PROMPT_LIMPIEZA = ChatPromptTemplate.from_messages([
    ('system', 'Corrige una transcripcion futbolistica en español. Conserva el significado, los tiempos, los nombres y los terminos de futbol. No inventes hechos ni elimines eventos. Devuelve solo el texto corregido.'),
    ('human', 'Texto a limpiar:\n{texto}')
])

def construir_cadena_llm():
    if not os.getenv('OPENAI_API_KEY'):
        raise RuntimeError('Para usar el modo llm configure OPENAI_API_KEY en el entorno.')
    modelo = ChatOpenAI(model='gpt-4o-mini', temperature=0)
    return PROMPT_LIMPIEZA | modelo | StrOutputParser()

cadena_demo = RunnableLambda(limpiar_texto_demo)

def limpiar_con_langchain(texto, modo='demo'):
    if modo == 'demo':
        return cadena_demo.invoke(texto)
    if modo == 'llm':
        return construir_cadena_llm().invoke({'texto': texto})
    raise ValueError("El modo debe ser 'demo' o 'llm'.")

df['texto_limpio'] = df['texto_original'].map(lambda x: limpiar_con_langchain(x, MODO_LIMPIEZA))
df[['tiempo_hms', 'texto_original', 'texto_limpio']].head(3)


,tiempo_hms,texto_original,texto_limpio
0,00:00:00,Gracias por estar ahí. Disfrutad del clásico m...,Gracias por estar ahí. Disfrutad del clásico m...
1,00:00:08,"Buenas noches, Jordi. Encantado.","Buenas noches, Jordi. Encantado."
2,00:00:10,"Bueno, pues aquí tenemos la alineación del Rea...","Bueno, pues aquí tenemos la alineación del Rea..."


### Limpieza con LangChain

Se implementan dos rutas con la misma interfaz. La ruta de demostración utiliza `RunnableLambda` y reglas deterministas para producir una salida reproducible sin API. La ruta opcional combina `ChatPromptTemplate`, `ChatOpenAI` y `StrOutputParser`; su instrucción obliga a conservar nombres, tiempos y expresiones futbolísticas. En esta fase no se invoca para evitar una dependencia externa y reducir cambios no verificables en la transcripción.


In [13]:
def seleccionar_tramo_partido(tabla):
    inicio = tabla['texto_limpio'].str.contains('arranca el partido', case=False, na=False).idxmax()
    fin = tabla['texto_limpio'].str.contains('Pita Sánchez Martínez', case=False, na=False).idxmax()
    salida = tabla.copy()
    salida['tramo'] = 'fuera_del_partido'
    salida['motivo_exclusion'] = 'Presentacion previa o comentario posterior al pitido final.'
    salida.loc[inicio:fin, 'tramo'] = 'partido'
    salida.loc[inicio:fin, 'motivo_exclusion'] = ''
    texto_inicio = salida.loc[inicio, 'texto_limpio']
    posicion = texto_inicio.lower().find('arranca el partido')
    salida.loc[inicio, 'texto_limpio'] = texto_inicio[posicion:].strip()
    return salida

df = seleccionar_tramo_partido(df)
partido_df = df[df['tramo'] == 'partido'].copy()
print(f'Lineas conservadas como partido: {len(partido_df)}')
print(f'Inicio: {partido_df.iloc[0].tiempo_hms} | Fin: {partido_df.iloc[-1].tiempo_hms}')
partido_df[['tiempo_hms', 'texto_limpio']].head(3)


Lineas conservadas como partido: 779
Inicio: 00:02:22 | Fin: 01:38:08


,tiempo_hms,texto_limpio
19,00:02:22,arranca el partido. Suerte para el Real Madrid...
20,00:02:29,primera aproximación se dejó el balón atrás Lu...
21,00:02:37,juego rival. Ha tocado ese balón. Es saque de ...


### Delimitación del contenido útil

Se conserva la narración desde la frase de inicio del partido hasta el momento en que el árbitro pita el final. Se excluye la presentación de alineaciones y el análisis posterior, porque no describen acciones ocurridas durante el juego. Esta decisión permite que los integrantes 2 y 3 trabajen sobre un corpus más representativo.


In [14]:
PROTEGIDOS = {
    'gol': r'\bgol(?:es)?\b',
    'tiro_de_esquina': r'saque de esquina|tiro de esquina|c[oó]rner',
    'saque_de_banda': r'saque de banda|lateral',
    'tarjeta_amarilla': r'tarjeta amarilla|cartulina amarilla|amarilla',
    'tarjeta_roja': r'tarjeta roja|cartulina roja|expuls',
    'fuera_de_juego': r'fuera de juego|fuera de lugar'
}

def contar_protegidos(texto, patrones=PROTEGIDOS):
    return {nombre: len(re.findall(patron, texto, flags=re.IGNORECASE)) for nombre, patron in patrones.items()}

original_partido = '\n'.join(partido_df['texto_original'])
limpio_partido = '\n'.join(partido_df['texto_limpio'])
comparacion = pd.DataFrame([contar_protegidos(original_partido), contar_protegidos(limpio_partido)], index=['original', 'limpio']).T
comparacion['diferencia'] = comparacion['limpio'] - comparacion['original']
comparacion


,original,limpio,diferencia
gol,83,83,0
tiro_de_esquina,18,18,0
saque_de_banda,23,23,0
tarjeta_amarilla,12,12,0
tarjeta_roja,1,1,0
fuera_de_juego,43,45,2


### Conservación de términos relevantes

Se comparan expresiones necesarias para el análisis posterior. Las correcciones se limitaron a nombres, puntuación, espacios y repeticiones evidentes; no se eliminaron las expresiones que describen goles, córners, saques de banda, tarjetas o fueras de juego.


In [15]:
COLUMNAS_SALIDA = ['linea', 'tiempo_hms', 'segundos', 'texto_original', 'texto_limpio', 'tramo', 'motivo_exclusion']

def escribir_lineas(tabla):
    return '\n'.join(f"{fila.tiempo_hms} | {fila.texto_limpio}" for fila in tabla.itertuples()) + '\n'

(BASE_DIR / 'narracion_limpia.txt').write_text(escribir_lineas(partido_df), encoding='utf-8')
muestra = partido_df[partido_df['segundos'] <= partido_df.iloc[0].segundos + 600]
(BASE_DIR / 'muestra_integrante_2.txt').write_text(escribir_lineas(muestra), encoding='utf-8')

EVENTOS = {
    'Goles': r'\bgol(?:azo|es)?\b|anota|marca',
    'Tiros de esquina': r'saque de esquina|tiro de esquina|c[oó]rner',
    'Saques de banda': r'saque de banda|lateral',
    'Faltas': r'\bfalta(?:s)?\b',
    'Tarjetas': r'cartulina|tarjeta|amarilla|roja|expuls',
    'Fuera de juego': r'fuera de juego|fuera de lugar|offside'
}

fragmentos = []
for evento, patron in EVENTOS.items():
    candidatos = partido_df[partido_df['texto_original'].str.contains(patron, case=False, regex=True, na=False)]
    if not candidatos.empty:
        fila = candidatos.iloc[0]
        fragmentos.append(f"[{evento}] {fila.tiempo_hms}\nOriginal: {fila.texto_original}\nLimpio: {fila.texto_limpio}\n")
(BASE_DIR / 'fragmentos_eventos_integrante_3.txt').write_text('\n'.join(fragmentos), encoding='utf-8')

df[COLUMNAS_SALIDA].to_csv(BASE_DIR / 'narracion_preparada.csv', index=False, encoding='utf-8')
(BASE_DIR / 'ficha_partido.json').write_text(json.dumps(PARTIDO, ensure_ascii=False, indent=2), encoding='utf-8')
print('Archivos de entrega generados correctamente.')


Archivos de entrega generados correctamente.


## Conclusiones de esta fase

En esta fase se confirma que la transcripción contiene 819 líneas con tiempos ordenados hasta 01:40:25. La principal dificultad fue la concatenación entre el código temporal y su descripción, problema que se resolvió separando el tiempo original, el formato `HH:MM:SS` y los segundos numéricos.

La limpieza corrigió errores evidentes de reconocimiento, espacios, puntuación y repeticiones, pero conservó el texto original y los términos futbolísticos necesarios. También se separó el contenido previo y posterior al partido para reducir ruido en las etapas siguientes.

El resultado de esta etapa es reproducible sin API gracias al modo demostración de LangChain. La ruta opcional con `ChatOpenAI` queda preparada para una revisión lingüística posterior, siempre que se configure la credencial fuera del notebook.
